# Basic RAG

This notebook demonstrates a basic Retrieval-Augmented Generation (RAG) flow for the GeForce NOW FAQ data. The flow covers creating an Elasticsearch text/vector index, searching the index for relevant FAQ entries, and using the search results to augment a language model's responses.

In [1]:
from elasticsearch import Elasticsearch
from constants import DEFAULT_ES_URL, DEFAULT_INDEX

client = Elasticsearch(DEFAULT_ES_URL)
if not client.ping():
    raise RuntimeError(f"Cannot connect to Elasticsearch at {DEFAULT_ES_URL}")

## Create text index

Create an Elasticsearch text index for the GeForce NOW FAQ data. Searchable fields are:
- question
- answer
- tag

In [3]:
from pathlib import Path
from faq_text_index import create_index, index_csv_data

data_path = Path().resolve().parent / "data" / "csv" / "geforce_now_faq.csv"

create_index(client, DEFAULT_INDEX, recreate=True)
inserted = index_csv_data(client, DEFAULT_INDEX, data_path)
print(f"Indexed {inserted} FAQ documents into '{DEFAULT_INDEX}' index.")

Indexed 98 FAQ documents into 'geforce-now-faq' index.


## Text Search

Search the text index for most relevant documents based on the given query/question.
By default no `boost_dict` is applied, meaning all fields have equal weight.

In [2]:
# Test index was created
client.get(index=DEFAULT_INDEX, id="1")
# client.search(index=DEFAULT_INDEX, query={"match": {"tag": "what is"}})

ObjectApiResponse({'_index': 'geforce-now-faq', '_id': '1', '_version': 1, '_seq_no': 0, '_primary_term': 1, 'found': True, '_source': {'id': '1', 'category': 'General Questions', 'tag': 'what-is-geforce-now', 'question': 'What is GeForce NOW?', 'answer': 'GeForce NOW is NVIDIA’s cloud- game streaming service, delivering real-time RTX-powered gameplay straight from the cloud to your laptop, desktop, Mac, Chromebook, SHIELD TV, select Samsung and LG TVs, iPhone, iPad, Android devices, Steam Deck, VR headsets, Linux PC and more. Connect to your favorite game store accounts and stream games you own, or check out hundreds of favorite free-to-play games. With cloud saves for supported games, you can pick up your game where you left off, on any supported device, wherever you are.'}})

In [3]:
from faq_text_search import search_faq

query = "What is Geforce Now?"
# boost_dict = {"question": 1, "tag": 1, "answer": 3}
results = search_faq(client, DEFAULT_INDEX, query)

for res in results:
    print(res["_source"])

{'id': '1', 'category': 'General Questions', 'tag': 'what-is-geforce-now', 'question': 'What is GeForce NOW?', 'answer': 'GeForce NOW is NVIDIA’s cloud- game streaming service, delivering real-time RTX-powered gameplay straight from the cloud to your laptop, desktop, Mac, Chromebook, SHIELD TV, select Samsung and LG TVs, iPhone, iPad, Android devices, Steam Deck, VR headsets, Linux PC and more. Connect to your favorite game store accounts and stream games you own, or check out hundreds of favorite free-to-play games. With cloud saves for supported games, you can pick up your game where you left off, on any supported device, wherever you are.'}
{'id': '52', 'category': 'Founders Memberships', 'tag': 'vip-founders-badge', 'question': 'What is the VIP Founders Member badge I now see in my account?', 'answer': 'To say thank you to the first to join and support GeForce NOW, Founders members will receive a special VIP Founders member badge in their account and in-app. An example of the Found